## Data split and preparation ##

In [1]:
import os
import random
import numpy as np
import pandas as pd
import torch
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import StratifiedKFold

In [2]:
def set_random_seed(seed: int = 42) -> None:
    """
    Set random seeds for reproducibility across numpy, PyTorch, and Python's random module.
    
    Parameters
    ----------
    seed : int
        Random seed value (default: 42)
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

# Set global seed
RANDOM_SEED = 42
set_random_seed(RANDOM_SEED)

In [4]:
def generate_folds(n_splits: int = 5, seed: int = 42) -> None:
    """
    Fetch the dataset, perform feature engineering, split into K-folds, 
    scale features, and save the datasets into local fold directories.
    """
    # Get the current working directory of the Jupyter Notebook
    base_dir = os.getcwd()
    
    # Fetch the dataset from UCI repository
    banknote = fetch_ucirepo(id=267)
    X = banknote.data.features.to_numpy()
    y = banknote.data.targets.to_numpy().ravel()
    
    # Feature engineering: calculate the interaction term (5th feature)
    variance = X[:, 0].reshape(-1, 1)
    skewness = X[:, 1].reshape(-1, 1)
    interaction = (variance * skewness).astype(np.float32)
    
    # Concatenate original features with the new interaction term
    X_expanded = np.hstack((X, interaction)).astype(np.float32)
    
    # Map labels from {0, 1} to {-1, 1} to match Pauli-Z observable range
    y_mapped = (2 * y - 1).astype(np.float32)

    # Initialize Stratified K-Fold cross-validator
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
    
    # Iterate through each fold
    for fold, (train_idx, test_idx) in enumerate(skf.split(X_expanded, y_mapped)):
        
        # --- DEFINITION OF FOLD DIRECTORY ADDED ---
        fold_dir = os.path.join(base_dir, f'fold_{fold + 1}')
        os.makedirs(fold_dir, exist_ok=True)
        # ------------------------------------------

        X_train, X_test = X_expanded[train_idx], X_expanded[test_idx]
        y_train, y_test = y_mapped[train_idx], y_mapped[test_idx]

        # --- DODANE SPRAWDZANIE BALANSU ---
        train_balance = pd.Series(y_train).value_counts(normalize=True).to_dict()
        test_balance = pd.Series(y_test).value_counts(normalize=True).to_dict()
        
        print(f"\nFold {fold + 1} Balance:")
        print(f"  Train -> Class -1.0: {train_balance.get(-1.0, 0):.1%}, Class 1.0: {train_balance.get(1.0, 0):.1%}")
        print(f"  Test  -> Class -1.0: {test_balance.get(-1.0, 0):.1%}, Class 1.0: {test_balance.get(1.0, 0):.1%}")
        
        # Fit the scaler on the training data, then transform both subsets
        scaler = MinMaxScaler(feature_range=(-np.pi/4, np.pi/4))
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)
        
        # Save the scaled training data and targets to CSV
        train_df = pd.DataFrame(X_train_scaled)
        train_df['target'] = y_train
        train_df.to_csv(os.path.join(fold_dir, 'train_data.csv'), index=False)
        
        # Save the scaled testing data and targets to CSV
        test_df = pd.DataFrame(X_test_scaled)
        test_df['target'] = y_test
        test_df.to_csv(os.path.join(fold_dir, 'test_data.csv'), index=False)
        
    print(f"\nSuccessfully generated {n_splits} folds in: {base_dir}")

# Execute the generation of folds
generate_folds(n_splits=5, seed=RANDOM_SEED)


Fold 1 Balance:
  Train -> Class -1.0: 55.5%, Class 1.0: 44.5%
  Test  -> Class -1.0: 55.6%, Class 1.0: 44.4%

Fold 2 Balance:
  Train -> Class -1.0: 55.5%, Class 1.0: 44.5%
  Test  -> Class -1.0: 55.6%, Class 1.0: 44.4%

Fold 3 Balance:
  Train -> Class -1.0: 55.6%, Class 1.0: 44.4%
  Test  -> Class -1.0: 55.5%, Class 1.0: 44.5%

Fold 4 Balance:
  Train -> Class -1.0: 55.6%, Class 1.0: 44.4%
  Test  -> Class -1.0: 55.5%, Class 1.0: 44.5%

Fold 5 Balance:
  Train -> Class -1.0: 55.6%, Class 1.0: 44.4%
  Test  -> Class -1.0: 55.5%, Class 1.0: 44.5%

Successfully generated 5 folds in: /home/raf/dev/axion/QC1/cross_validation/Data
